<a href="https://colab.research.google.com/github/teklemaryamas-et/HWRM_Thesis_2026/blob/main/CHIRPS_Bias_correction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# CONVERT DAILY 9-STATION RAINFALL EXCEL TO MONTHLY RAINFALL
# Corrected for sheet names like AS_Precip and coordinate names like ASPCP
# ============================================================

!pip install pandas openpyxl -q

import pandas as pd
import numpy as np
from google.colab import files

# ============================================================
# 1. UPLOAD TWO EXCEL FILES
# ============================================================

print("Upload TWO Excel files:")
print("1) Daily rainfall Excel file with 9 station sheets")
print("2) Station coordinate Excel file with NAME, Lat, Long, ELEVATION")

uploaded = files.upload()

excel_files = [
    f for f in uploaded.keys()
    if f.lower().endswith((".xlsx", ".xls"))
]

if len(excel_files) < 2:
    raise FileNotFoundError("Please upload both Excel files.")

print("\nUploaded files:")
for i, f in enumerate(excel_files):
    print(i + 1, f)

# ============================================================
# 2. IDENTIFY DAILY AND COORDINATE FILES
# ============================================================

daily_file = None
coord_file = None

for f in excel_files:
    fname = f.lower()
    if "station" in fname or "9stations" in fname or "coord" in fname:
        coord_file = f
    else:
        daily_file = f

if daily_file is None or coord_file is None:
    # Based on your uploaded file names
    coord_file = "9Stations.xlsx"
    daily_file = [f for f in excel_files if f != coord_file][0]

print("\nDaily rainfall file used:")
print(daily_file)

print("\nCoordinate file used:")
print(coord_file)

# ============================================================
# 3. READ AND STANDARDIZE COORDINATE FILE
# ============================================================

coord_df = pd.read_excel(coord_file)
coord_df.columns = [str(c).strip() for c in coord_df.columns]

print("\nCoordinate columns:")
print(coord_df.columns.tolist())

# Rename columns based on your file
coord_df = coord_df.rename(columns={
    "NAME": "station_code",
    "Name": "station_code",
    "name": "station_code",
    "Lat": "lat",
    "LAT": "lat",
    "latitude": "lat",
    "Long": "lon",
    "LONG": "lon",
    "Lon": "lon",
    "longitude": "lon",
    "ELEVATION": "elevation",
    "Elevation": "elevation",
    "elevation": "elevation"
})

required_coord_cols = ["station_code", "lat", "lon"]

missing = [c for c in required_coord_cols if c not in coord_df.columns]
if len(missing) > 0:
    raise ValueError(f"Coordinate file missing columns: {missing}")

if "elevation" not in coord_df.columns:
    coord_df["elevation"] = np.nan

coord_df = coord_df[["station_code", "lat", "lon", "elevation"]].copy()

coord_df["station_code"] = coord_df["station_code"].astype(str).str.strip()
coord_df["lat"] = pd.to_numeric(coord_df["lat"], errors="coerce")
coord_df["lon"] = pd.to_numeric(coord_df["lon"], errors="coerce")
coord_df["elevation"] = pd.to_numeric(coord_df["elevation"], errors="coerce")

coord_df = coord_df.dropna(subset=["station_code", "lat", "lon"])

print("\nCoordinate table:")
display(coord_df)

# ============================================================
# 4. MANUAL MATCHING BETWEEN SHEET NAMES AND COORDINATE CODES
# ============================================================

sheet_to_coord_code = {
    "AS_Precip": "ASPCP",
    "BD_Precip": "BDPCP",
    "DBR_Precip": "DBRPCP",
    "DM_Precip": "DMPCP",
    "DS_Precip": "DSPCP",
    "DT_Precip": "DTPCP",
    "FS_Precip": "FSPCP",
    "GN_Precip": "GNPCP",
    "NK_Precip": "NKPCP"
}

# Optional readable station names
station_full_names = {
    "ASPCP": "Assosa",
    "BDPCP": "Bahir Dar",
    "DBRPCP": "Debre Birhan",
    "DMPCP": "Debre Markos",
    "DSPCP": "Dessie",
    "DTPCP": "Debre Tabor",
    "FSPCP": "Finote Selam",
    "GNPCP": "Gondar",
    "NKPCP": "Nekemte"
}

# ============================================================
# 5. READ DAILY EXCEL SHEETS
# ============================================================

xls = pd.ExcelFile(daily_file)
sheet_names = xls.sheet_names

print("\nSheets found:")
print(sheet_names)

all_monthly = []
process_log = []

# ============================================================
# 6. FUNCTION TO STANDARDIZE DAILY RAINFALL SHEET
# ============================================================

def standardize_daily_sheet(df, sheet_name):

    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]

    lower_cols = {c.lower(): c for c in df.columns}

    rename_map = {}

    # Date column
    for key in ["date", "time", "day"]:
        if key in lower_cols:
            rename_map[lower_cols[key]] = "Date"
            break

    # Rainfall column
    for key in [
        "obs_rainfall", "rainfall", "rain", "precip",
        "precipitation", "rf", "rr", "value"
    ]:
        if key in lower_cols:
            rename_map[lower_cols[key]] = "obs_rainfall"
            break

    df = df.rename(columns=rename_map)

    if "Date" not in df.columns or "obs_rainfall" not in df.columns:
        print(f"\nSheet '{sheet_name}' columns:")
        print(df.columns.tolist())
        raise ValueError(
            f"Sheet '{sheet_name}' must contain Date and rainfall column."
        )

    df = df[["Date", "obs_rainfall"]].copy()

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df["obs_rainfall"] = pd.to_numeric(df["obs_rainfall"], errors="coerce")

    df = df.dropna(subset=["Date", "obs_rainfall"])

    # Remove negative rainfall values
    df = df[df["obs_rainfall"] >= 0].copy()

    return df

# ============================================================
# 7. CONVERT EACH SHEET FROM DAILY TO MONTHLY
# ============================================================

for sheet in sheet_names:

    print("\nProcessing sheet:", sheet)

    if sheet not in sheet_to_coord_code:
        print("Skipped sheet because it is not in sheet_to_coord_code:", sheet)
        continue

    station_code = sheet_to_coord_code[sheet]

    coord_match = coord_df[coord_df["station_code"] == station_code]

    if len(coord_match) == 0:
        print("No coordinate found for station code:", station_code)
        continue

    coord_match = coord_match.iloc[0]

    station_name = station_full_names.get(station_code, station_code)
    lon = coord_match["lon"]
    lat = coord_match["lat"]
    elevation = coord_match["elevation"]

    daily_df = pd.read_excel(daily_file, sheet_name=sheet)
    daily_df = standardize_daily_sheet(daily_df, sheet)

    daily_df["year"] = daily_df["Date"].dt.year
    daily_df["month"] = daily_df["Date"].dt.month

    monthly_df = (
        daily_df
        .groupby(["year", "month"], as_index=False)
        .agg(obs_rainfall=("obs_rainfall", "sum"))
    )

    monthly_df["station"] = station_name
    monthly_df["station_code"] = station_code
    monthly_df["lon"] = lon
    monthly_df["lat"] = lat
    monthly_df["elevation"] = elevation

    monthly_df = monthly_df[
        ["station", "station_code", "lon", "lat", "elevation", "year", "month", "obs_rainfall"]
    ]

    all_monthly.append(monthly_df)

    process_log.append({
        "sheet": sheet,
        "station_code": station_code,
        "station": station_name,
        "records_daily": len(daily_df),
        "records_monthly": len(monthly_df),
        "start_year": monthly_df["year"].min(),
        "end_year": monthly_df["year"].max()
    })

    print(
        f"Done: {station_name} ({station_code}) | "
        f"daily records: {len(daily_df)} | monthly records: {len(monthly_df)}"
    )

# ============================================================
# 8. COMBINE AND SAVE OUTPUT
# ============================================================

if len(all_monthly) == 0:
    raise ValueError("No station sheets were processed. Check sheet names and matching dictionary.")

monthly_all = pd.concat(all_monthly, ignore_index=True)

monthly_all = monthly_all.sort_values(
    ["station", "year", "month"]
).reset_index(drop=True)

process_log_df = pd.DataFrame(process_log)

print("\nCombined monthly rainfall data:")
display(monthly_all.head(30))

print("\nProcessing summary:")
display(process_log_df)

print("\nStations processed:", monthly_all["station"].nunique())
print("Total monthly records:", len(monthly_all))
print("Year range:", monthly_all["year"].min(), "-", monthly_all["year"].max())

print("\nMonthly records per station:")
display(monthly_all.groupby(["station", "station_code"]).size().reset_index(name="monthly_records"))

# Save final files
output_csv = "NMA_station_monthly_rainfall_COMBINED.csv"
output_excel = "NMA_station_monthly_rainfall_COMBINED.xlsx"
log_csv = "NMA_daily_to_monthly_conversion_log.csv"

monthly_all.to_csv(output_csv, index=False)
monthly_all.to_excel(output_excel, index=False)
process_log_df.to_csv(log_csv, index=False)

print("\nSaved output files:")
print(output_csv)
print(output_excel)
print(log_csv)

files.download(output_csv)
files.download(output_excel)
files.download(log_csv)

Upload TWO Excel files:
1) Daily rainfall Excel file with 9 station sheets
2) Station coordinate Excel file with NAME, Lat, Long, ELEVATION


Saving 9Stations.xlsx to 9Stations (1).xlsx
Saving Precip_9St daily.xlsx to Precip_9St daily (2).xlsx

Uploaded files:
1 9Stations (1).xlsx
2 Precip_9St daily (2).xlsx

Daily rainfall file used:
Precip_9St daily (2).xlsx

Coordinate file used:
9Stations (1).xlsx

Coordinate columns:
['ID', 'NAME', 'Lat', 'Long', 'ELEVATION']

Coordinate table:


,station_code,lat,lon,elevation
0,ASPCP,10.046,34.546,1541
1,BDPCP,11.595,37.360,1800
2,DBRPCP,9.670,39.513,3206
3,DMPCP,10.326,37.739,2446
4,DSPCP,11.118,39.635,2553
5,DTPCP,11.867,37.995,2612
6,FSPCP,10.682,37.263,1872
7,GNPCP,12.521,37.432,1973
8,NKPCP,9.083,36.549,2119



Sheets found:
['AS_Precip', 'BD_Precip', 'DBR_Precip', 'DM_Precip', 'DS_Precip', 'DT_Precip', 'FS_Precip', 'GN_Precip', 'NK_Precip']

Processing sheet: AS_Precip
Done: Assosa (ASPCP) | daily records: 9862 | monthly records: 324

Processing sheet: BD_Precip
Done: Bahir Dar (BDPCP) | daily records: 9862 | monthly records: 324

Processing sheet: DBR_Precip
Done: Debre Birhan (DBRPCP) | daily records: 9862 | monthly records: 324

Processing sheet: DM_Precip
Done: Debre Markos (DMPCP) | daily records: 9862 | monthly records: 324

Processing sheet: DS_Precip
Done: Dessie (DSPCP) | daily records: 9862 | monthly records: 324

Processing sheet: DT_Precip
Done: Debre Tabor (DTPCP) | daily records: 9862 | monthly records: 324

Processing sheet: FS_Precip
Done: Finote Selam (FSPCP) | daily records: 9862 | monthly records: 324

Processing sheet: GN_Precip
Done: Gondar (GNPCP) | daily records: 9862 | monthly records: 324

Processing sheet: NK_Precip
Done: Nekemte (NKPCP) | daily records: 9862 | mon

,station,station_code,lon,lat,elevation,year,month,obs_rainfall
0,Assosa,ASPCP,34.546,10.046,1541,1994,1,0.0
1,Assosa,ASPCP,34.546,10.046,1541,1994,2,0.0
2,Assosa,ASPCP,34.546,10.046,1541,1994,3,0.0
3,Assosa,ASPCP,34.546,10.046,1541,1994,4,0.0
4,Assosa,ASPCP,34.546,10.046,1541,1994,5,130.0
5,Assosa,ASPCP,34.546,10.046,1541,1994,6,152.5
6,Assosa,ASPCP,34.546,10.046,1541,1994,7,258.9
7,Assosa,ASPCP,34.546,10.046,1541,1994,8,170.6
8,Assosa,ASPCP,34.546,10.046,1541,1994,9,142.4
9,Assosa,ASPCP,34.546,10.046,1541,1994,10,106.3


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# GOOGLE COLAB CODE
# CHIRPS BIAS CORRECTION USING 4 MACHINE LEARNING MODELS
# Models: SVM, Random Forest, XGBoost, KNN
# Selection: multi-criteria with stronger weight for bias correction
# ============================================================

# ============================================================
# 1. INSTALL PACKAGES
# ============================================================

!pip install rasterio xgboost scikit-learn pandas numpy openpyxl -q

# ============================================================
# 2. IMPORT LIBRARIES
# ============================================================

import os
import re
import glob
import zipfile
import warnings
import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import rowcol
from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

# ============================================================
# 3. UPLOAD ZIP FILE
# ============================================================

print("Upload your zipped data folder now...")
uploaded = files.upload()

zip_files = [f for f in uploaded.keys() if f.lower().endswith(".zip")]

if len(zip_files) == 0:
    raise FileNotFoundError("No zip file uploaded. Please upload a zipped folder containing CHIRPS and NMA data.")

zip_file = zip_files[0]
print("Uploaded zip file:", zip_file)

# ============================================================
# 4. UNZIP DATA
# ============================================================

base_dir = "/content/HWRM_ML_Bias_Correction"
data_dir = os.path.join(base_dir, "Input_Data")
output_folder = os.path.join(base_dir, "CHIRPS_ML_Bias_Correction_Output")

os.makedirs(data_dir, exist_ok=True)
os.makedirs(output_folder, exist_ok=True)
os.makedirs(os.path.join(output_folder, "Tables"), exist_ok=True)
os.makedirs(os.path.join(output_folder, "Corrected_Rasters"), exist_ok=True)

with zipfile.ZipFile(zip_file, "r") as zip_ref:
    zip_ref.extractall(data_dir)

print("Data extracted to:", data_dir)

# ============================================================
# 5. AUTOMATICALLY FIND CHIRPS FOLDER AND STATION FILE
# ============================================================

all_tif_files = glob.glob(os.path.join(data_dir, "**", "*.tif"), recursive=True) + \
                glob.glob(os.path.join(data_dir, "**", "*.tiff"), recursive=True)

if len(all_tif_files) == 0:
    raise FileNotFoundError("No CHIRPS GeoTIFF files found inside the uploaded zip.")

# Use the parent folder containing most tif files as CHIRPS folder
tif_parent_counts = {}
for f in all_tif_files:
    parent = os.path.dirname(f)
    tif_parent_counts[parent] = tif_parent_counts.get(parent, 0) + 1

chirps_folder = max(tif_parent_counts, key=tif_parent_counts.get)

print("Detected CHIRPS folder:")
print(chirps_folder)
print("Number of tif files found:", len(all_tif_files))

# Find station CSV or Excel file
station_files = glob.glob(os.path.join(data_dir, "**", "*.csv"), recursive=True) + \
                glob.glob(os.path.join(data_dir, "**", "*.xlsx"), recursive=True) + \
                glob.glob(os.path.join(data_dir, "**", "*.xls"), recursive=True)

if len(station_files) == 0:
    raise FileNotFoundError("No station rainfall CSV/Excel file found inside the uploaded zip.")

# Prefer file names containing NMA or station
preferred_station_files = [
    f for f in station_files
    if ("nma" in os.path.basename(f).lower()) or ("station" in os.path.basename(f).lower())
]

if len(preferred_station_files) > 0:
    station_file = preferred_station_files[0]
else:
    station_file = station_files[0]

print("Detected station rainfall file:")
print(station_file)

# ============================================================
# 6. SETTINGS
# ============================================================

start_year = 1994
end_year = 2014

test_size = 0.30
random_state = 42

# bias_correction = gives stronger importance to bias
# prediction = equal weight to all metrics
selection_mode = "bias_correction"

# ============================================================
# 7. HELPER FUNCTIONS
# ============================================================

def extract_year_month_from_filename(file_path):
    fname = os.path.basename(file_path)

    match1 = re.search(r"(19\d{2}|20\d{2})[_\-.](0[1-9]|1[0-2])", fname)
    if match1:
        return int(match1.group(1)), int(match1.group(2))

    match2 = re.search(r"(19\d{2}|20\d{2})(0[1-9]|1[0-2])", fname)
    if match2:
        return int(match2.group(1)), int(match2.group(2))

    return None, None


def read_station_data(file_path):
    if file_path.lower().endswith(".csv"):
        df = pd.read_csv(file_path)
    else:
        df = pd.read_excel(file_path)

    df.columns = [str(c).strip() for c in df.columns]
    return df


def standardize_station_columns(df):
    """
    Required final columns:
    station, lon, lat, year, month, obs_rainfall
    """

    lower_cols = {c.lower().strip(): c for c in df.columns}
    col_map = {}

    for key in ["station", "station_name", "name", "site"]:
        if key in lower_cols:
            col_map[lower_cols[key]] = "station"
            break

    for key in ["lon", "longitude", "x"]:
        if key in lower_cols:
            col_map[lower_cols[key]] = "lon"
            break

    for key in ["lat", "latitude", "y"]:
        if key in lower_cols:
            col_map[lower_cols[key]] = "lat"
            break

    for key in ["year", "yr"]:
        if key in lower_cols:
            col_map[lower_cols[key]] = "year"
            break

    for key in ["month", "mon", "mn"]:
        if key in lower_cols:
            col_map[lower_cols[key]] = "month"
            break

    for key in ["obs_rainfall", "rainfall", "rain", "precip", "precipitation", "nma", "observed", "obs"]:
        if key in lower_cols:
            col_map[lower_cols[key]] = "obs_rainfall"
            break

    df = df.rename(columns=col_map)

    required = ["station", "lon", "lat", "year", "month", "obs_rainfall"]
    missing = [c for c in required if c not in df.columns]

    if len(missing) > 0:
        print("Your station file columns are:")
        print(list(df.columns))
        raise ValueError(
            f"Missing columns: {missing}\n"
            "Please rename station file columns to:\n"
            "station, lon, lat, year, month, obs_rainfall"
        )

    df = df[required].copy()

    df["year"] = pd.to_numeric(df["year"], errors="coerce")
    df["month"] = pd.to_numeric(df["month"], errors="coerce")
    df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
    df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
    df["obs_rainfall"] = pd.to_numeric(df["obs_rainfall"], errors="coerce")

    df = df.dropna(subset=["station", "lon", "lat", "year", "month", "obs_rainfall"])

    df["year"] = df["year"].astype(int)
    df["month"] = df["month"].astype(int)

    df = df[(df["month"] >= 1) & (df["month"] <= 12)]
    df = df[df["obs_rainfall"] >= 0].copy()

    df["date_id"] = df["year"].astype(str) + "-" + df["month"].astype(str).str.zfill(2)

    return df


def extract_raster_value_at_point(raster_path, lon, lat):
    try:
        with rasterio.open(raster_path) as src:
            row, col = rowcol(src.transform, lon, lat)

            if row < 0 or col < 0 or row >= src.height or col >= src.width:
                return np.nan

            value = src.read(1)[row, col]

            if src.nodata is not None and value == src.nodata:
                return np.nan

            return float(value)

    except Exception:
        return np.nan


def calculate_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    if np.std(y_true) == 0 or np.std(y_pred) == 0:
        corr = np.nan
    else:
        corr = np.corrcoef(y_true, y_pred)[0, 1]

    bias = np.mean(y_pred - y_true)

    return rmse, mae, r2, corr, bias


def add_month_features(df):
    df = df.copy()
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    return df


def normalize_metric(series, higher_is_better=True):
    s = series.astype(float)

    if s.max() == s.min():
        return pd.Series(np.zeros(len(s)), index=s.index)

    if higher_is_better:
        return (s.max() - s) / (s.max() - s.min())
    else:
        return (s - s.min()) / (s.max() - s.min())


# ============================================================
# 8. CREATE CHIRPS INVENTORY
# ============================================================

chirps_files = sorted(
    glob.glob(os.path.join(chirps_folder, "**", "*.tif"), recursive=True) +
    glob.glob(os.path.join(chirps_folder, "**", "*.tiff"), recursive=True)
)

chirps_inventory = []

for f in chirps_files:
    year, month = extract_year_month_from_filename(f)

    if year is not None and month is not None:
        chirps_inventory.append({
            "year": year,
            "month": month,
            "date_id": f"{year}-{month:02d}",
            "file": f
        })

chirps_inventory = pd.DataFrame(chirps_inventory)

chirps_inventory = chirps_inventory[
    (chirps_inventory["year"] >= start_year) &
    (chirps_inventory["year"] <= end_year)
].copy()

chirps_inventory = chirps_inventory.sort_values(["year", "month"]).reset_index(drop=True)

print("CHIRPS files in 1994–2014:", len(chirps_inventory))
display(chirps_inventory.head())

if len(chirps_inventory) == 0:
    raise ValueError("No CHIRPS files found for 1994–2014. Check file names contain year and month.")

chirps_inventory.to_csv(
    os.path.join(output_folder, "Tables", "CHIRPS_inventory_1994_2014.csv"),
    index=False
)

# ============================================================
# 9. READ STATION RAINFALL DATA
# ============================================================

station_raw = read_station_data(station_file)
station_df = standardize_station_columns(station_raw)

station_df = station_df[
    (station_df["year"] >= start_year) &
    (station_df["year"] <= end_year)
].copy()

print("Station records in 1994–2014:", len(station_df))
display(station_df.head())

station_df.to_csv(
    os.path.join(output_folder, "Tables", "NMA_station_data_used.csv"),
    index=False
)

# ============================================================
# 10. EXTRACT CHIRPS VALUES AT STATION LOCATIONS
# ============================================================

print("Extracting CHIRPS values at station locations...")

file_lookup = dict(zip(chirps_inventory["date_id"], chirps_inventory["file"]))

records = []

for _, row in station_df.iterrows():

    date_id = row["date_id"]

    if date_id not in file_lookup:
        continue

    chirps_value = extract_raster_value_at_point(
        raster_path=file_lookup[date_id],
        lon=row["lon"],
        lat=row["lat"]
    )

    records.append({
        "station": row["station"],
        "lon": row["lon"],
        "lat": row["lat"],
        "year": row["year"],
        "month": row["month"],
        "date_id": date_id,
        "chirps": chirps_value,
        "obs_rainfall": row["obs_rainfall"]
    })

ml_df = pd.DataFrame(records)

ml_df = ml_df.replace([np.inf, -np.inf], np.nan)
ml_df = ml_df.dropna(subset=["chirps", "obs_rainfall", "lon", "lat"])
ml_df = ml_df[(ml_df["chirps"] >= 0) & (ml_df["obs_rainfall"] >= 0)].copy()
ml_df = add_month_features(ml_df)

print("Final station-matched ML dataset records:", len(ml_df))
display(ml_df.head())

if len(ml_df) < 50:
    raise ValueError(
        "Too few station-matched records. Check station coordinates, station dates, and CHIRPS file names."
    )

ml_df.to_csv(
    os.path.join(output_folder, "Tables", "ML_training_dataset_CHIRPS_NMA.csv"),
    index=False
)

# ============================================================
# 11. DEFINE FEATURES AND TARGET
# ============================================================

feature_cols = ["chirps", "lon", "lat", "month_sin", "month_cos"]
target_col = "obs_rainfall"

X = ml_df[feature_cols]
y = ml_df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=test_size,
    random_state=random_state
)

# ============================================================
# 12. DEFINE ML MODELS
# ============================================================

models = {
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVR(kernel="rbf", C=10, gamma="scale", epsilon=0.1))
    ]),

    "Random Forest": RandomForestRegressor(
        n_estimators=500,
        random_state=random_state,
        max_features="sqrt",
        min_samples_leaf=2,
        n_jobs=-1
    ),

    "XGBoost": XGBRegressor(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=random_state,
        n_jobs=-1
    ),

    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsRegressor(n_neighbors=5, weights="distance"))
    ])
}

# ============================================================
# 13. TRAIN AND EVALUATE MODELS
# ============================================================

results = []
trained_models = {}

for name, model in models.items():

    print("Training:", name)

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_pred = np.maximum(y_pred, 0)

    rmse, mae, r2, corr, bias = calculate_metrics(y_test, y_pred)

    results.append({
        "Model": name,
        "RMSE_mm": rmse,
        "MAE_mm": mae,
        "R2": r2,
        "Correlation": corr,
        "Bias_mm": bias,
        "Abs_Bias_mm": abs(bias)
    })

    trained_models[name] = model

results_df = pd.DataFrame(results)

# ============================================================
# 14. MULTI-CRITERIA MODEL SELECTION
# ============================================================

selection_df = results_df.copy()

selection_df["RMSE_norm"] = normalize_metric(selection_df["RMSE_mm"], higher_is_better=False)
selection_df["MAE_norm"] = normalize_metric(selection_df["MAE_mm"], higher_is_better=False)
selection_df["R2_norm"] = normalize_metric(selection_df["R2"], higher_is_better=True)
selection_df["Correlation_norm"] = normalize_metric(selection_df["Correlation"], higher_is_better=True)
selection_df["Bias_norm"] = normalize_metric(selection_df["Abs_Bias_mm"], higher_is_better=False)

if selection_mode == "bias_correction":
    weights = {
        "RMSE_norm": 0.20,
        "MAE_norm": 0.15,
        "R2_norm": 0.15,
        "Correlation_norm": 0.10,
        "Bias_norm": 0.40
    }
elif selection_mode == "prediction":
    weights = {
        "RMSE_norm": 0.20,
        "MAE_norm": 0.20,
        "R2_norm": 0.20,
        "Correlation_norm": 0.20,
        "Bias_norm": 0.20
    }
else:
    raise ValueError("selection_mode must be 'bias_correction' or 'prediction'.")

selection_df["Final_Selection_Score"] = 0

for col, w in weights.items():
    selection_df["Final_Selection_Score"] += selection_df[col] * w

selection_df = selection_df.sort_values("Final_Selection_Score").reset_index(drop=True)
selection_df["Final_Selection_Rank"] = np.arange(1, len(selection_df) + 1)

selected_model_name = selection_df.loc[0, "Model"]
selected_model = trained_models[selected_model_name]

print("================================================")
print("MODEL PERFORMANCE AND FINAL SELECTION")
print("================================================")
display(selection_df[[
    "Model", "RMSE_mm", "MAE_mm", "R2", "Correlation",
    "Bias_mm", "Abs_Bias_mm", "Final_Selection_Score",
    "Final_Selection_Rank"
]])

print("Selected model:", selected_model_name)

selection_df.to_csv(
    os.path.join(output_folder, "Tables", "ML_model_performance_and_final_selection.csv"),
    index=False
)

# ============================================================
# 15. TRAIN SELECTED MODEL USING ALL DATA
# ============================================================

selected_model.fit(X, y)

# ============================================================
# 16. APPLY SELECTED MODEL TO ALL CHIRPS RASTERS
# ============================================================

print("Applying selected model to all CHIRPS rasters...")

all_chirps_files = sorted(
    glob.glob(os.path.join(chirps_folder, "**", "*.tif"), recursive=True) +
    glob.glob(os.path.join(chirps_folder, "**", "*.tiff"), recursive=True)
)

correction_log = []

for f in all_chirps_files:

    year, month = extract_year_month_from_filename(f)

    if year is None or month is None:
        continue

    with rasterio.open(f) as src:

        arr = src.read(1).astype("float32")
        profile = src.profile.copy()
        nodata = src.nodata
        transform = src.transform

        valid_mask = np.isfinite(arr)

        if nodata is not None:
            valid_mask = valid_mask & (arr != nodata)

        valid_mask = valid_mask & (arr >= 0)

        corrected = np.full(arr.shape, np.nan, dtype="float32")

        if valid_mask.sum() > 0:

            rows, cols = np.where(valid_mask)

            xs, ys = rasterio.transform.xy(transform, rows, cols)
            xs = np.array(xs)
            ys = np.array(ys)

            chirps_vals = arr[valid_mask]

            month_sin = np.sin(2 * np.pi * month / 12)
            month_cos = np.cos(2 * np.pi * month / 12)

            pred_df = pd.DataFrame({
                "chirps": chirps_vals,
                "lon": xs,
                "lat": ys,
                "month_sin": month_sin,
                "month_cos": month_cos
            })

            pred_corrected = selected_model.predict(pred_df[feature_cols])
            pred_corrected = np.maximum(pred_corrected, 0)

            corrected[valid_mask] = pred_corrected.astype("float32")

        profile.update(
            dtype="float32",
            nodata=-9999,
            compress="lzw"
        )

        corrected_out = np.where(np.isfinite(corrected), corrected, -9999).astype("float32")

        out_name = f"{selected_model_name.replace(' ', '_')}_corrected_{year}{month:02d}.tif"
        out_path = os.path.join(output_folder, "Corrected_Rasters", out_name)

        with rasterio.open(out_path, "w", **profile) as dst:
            dst.write(corrected_out, 1)

        correction_log.append({
            "year": year,
            "month": month,
            "date_id": f"{year}-{month:02d}",
            "input_file": f,
            "output_file": out_path,
            "selected_model": selected_model_name,
            "valid_pixels": int(valid_mask.sum()),
            "mean_raw_CHIRPS": float(np.nanmean(arr[valid_mask])) if valid_mask.sum() > 0 else np.nan,
            "mean_corrected": float(np.nanmean(corrected[valid_mask])) if valid_mask.sum() > 0 else np.nan
        })

correction_log_df = pd.DataFrame(correction_log)

correction_log_df.to_csv(
    os.path.join(output_folder, "Tables", "CHIRPS_bias_correction_log.csv"),
    index=False
)

# ============================================================
# 17. ZIP OUTPUTS FOR DOWNLOAD
# ============================================================

summary_text = f"""
CHIRPS rainfall bias correction was performed using four machine-learning models:
Support Vector Machine, Random Forest, XGBoost, and K-Nearest Neighbors. NMA station
rainfall was used as the reference dataset. CHIRPS rainfall values were extracted at
station locations and combined with longitude, latitude, and seasonal month features.

The models were evaluated using RMSE, MAE, R2, correlation, and bias. Final model
selection was based on {selection_mode} multi-criteria scoring. Since the objective was
bias correction, systematic bias was given stronger consideration along with error and
correlation metrics. The selected model was {selected_model_name}. The selected model
was then trained using all station-matched records and applied to all CHIRPS raster files
to produce the final bias-corrected CHIRPS rainfall dataset.
"""

with open(os.path.join(output_folder, "Tables", "ML_bias_correction_method_summary.txt"), "w") as f:
    f.write(summary_text)

zip_output = "/content/CHIRPS_ML_Bias_Correction_Output.zip"

if os.path.exists(zip_output):
    os.remove(zip_output)

import shutil
shutil.make_archive(
    base_name=zip_output.replace(".zip", ""),
    format="zip",
    root_dir=output_folder
)

# ============================================================
# 18. FINAL MESSAGE AND DOWNLOAD
# ============================================================

print("================================================")
print("DONE SUCCESSFULLY")
print("================================================")
print("Selected model:", selected_model_name)

print("\nImportant tables:")
print(os.path.join(output_folder, "Tables", "ML_model_performance_and_final_selection.csv"))
print(os.path.join(output_folder, "Tables", "ML_training_dataset_CHIRPS_NMA.csv"))
print(os.path.join(output_folder, "Tables", "CHIRPS_bias_correction_log.csv"))

print("\nCorrected rasters folder:")
print(os.path.join(output_folder, "Corrected_Rasters"))

print("\nOutput zip file:")
print(zip_output)

files.download(zip_output)

Upload your zipped data folder now...


Saving Bias-correction_Input_data.zip to Bias-correction_Input_data.zip
Uploaded zip file: Bias-correction_Input_data.zip
Data extracted to: /content/HWRM_ML_Bias_Correction/Input_Data
Detected CHIRPS folder:
/content/HWRM_ML_Bias_Correction/Input_Data/Bias-correction_Input_data/CHIRPS_Monthly_Abay
Number of tif files found: 420
Detected station rainfall file:
/content/HWRM_ML_Bias_Correction/Input_Data/Bias-correction_Input_data/NMA_station_monthly_rainfall_COMBINED.csv
CHIRPS files in 1994–2014: 252


,year,month,date_id,file
0,1994,1,1994-01,/content/HWRM_ML_Bias_Correction/Input_Data/Bi...
1,1994,2,1994-02,/content/HWRM_ML_Bias_Correction/Input_Data/Bi...
2,1994,3,1994-03,/content/HWRM_ML_Bias_Correction/Input_Data/Bi...
3,1994,4,1994-04,/content/HWRM_ML_Bias_Correction/Input_Data/Bi...
4,1994,5,1994-05,/content/HWRM_ML_Bias_Correction/Input_Data/Bi...


Station records in 1994–2014: 2268


,station,lon,lat,year,month,obs_rainfall,date_id
0,Assosa,34.546,10.046,1994,1,0.0,1994-01
1,Assosa,34.546,10.046,1994,2,0.0,1994-02
2,Assosa,34.546,10.046,1994,3,0.0,1994-03
3,Assosa,34.546,10.046,1994,4,0.0,1994-04
4,Assosa,34.546,10.046,1994,5,130.0,1994-05


...

Training: SVM
Training: Random Forest
Training: XGBoost
Training: KNN
MODEL PERFORMANCE AND FINAL SELECTION


,Model,RMSE_mm,MAE_mm,R2,Correlation,Bias_mm,Abs_Bias_mm,Final_Selection_Score,Final_Selection_Rank
0,XGBoost,50.860834,30.468261,0.852644,0.924966,3.045336,3.045336,0.104066,1
1,Random Forest,51.021026,30.458898,0.851714,0.924523,3.305149,3.305149,0.149937,2
2,SVM,50.959166,30.366090,0.852074,0.926206,-5.834973,5.834973,0.408451,3
3,KNN,54.868865,31.990518,0.828505,0.913857,2.297178,2.297178,0.600000,4


Selected model: XGBoost
Applying selected model to all CHIRPS rasters...
DONE SUCCESSFULLY
Selected model: XGBoost

Important tables:
/content/HWRM_ML_Bias_Correction/CHIRPS_ML_Bias_Correction_Output/Tables/ML_model_performance_and_final_selection.csv
/content/HWRM_ML_Bias_Correction/CHIRPS_ML_Bias_Correction_Output/Tables/ML_training_dataset_CHIRPS_NMA.csv
/content/HWRM_ML_Bias_Correction/CHIRPS_ML_Bias_Correction_Output/Tables/CHIRPS_bias_correction_log.csv

Corrected rasters folder:
/content/HWRM_ML_Bias_Correction/CHIRPS_ML_Bias_Correction_Output/Corrected_Rasters

Output zip file:
/content/CHIRPS_ML_Bias_Correction_Output.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>